# 🔤 Brahmi Script OCR — TrOCR Training on Kaggle

This notebook trains TrOCR on a **map.json-driven Brahmi dataset** using Kaggle's dual T4 GPUs.

### ⚡ CRITICAL Setup Instructions for Overnight Training:
1. **Turn on the GPU:** Go to **Notebook Options (right panel) → Accelerator → select `GPU T4 x2`**.
2. **Add Your Dataset:** Click **Add Data** (top right) → **Upload Data**.
   - Give it a title like `brahmi-curated-dataset`.
   - Upload your `curated_dataset.zip` file. (Kaggle will automatically unzip it for you!).
3. **Turn on Internet:** Go to **Notebook Options → Internet on** (Required to download the Microsoft model).
4. **To Train Overnight:** At the top right, do **NOT** click 'Run All'. Instead, click **Save Version → Save & Run All (Commit)**.
   - This runs the notebook in the background on Kaggle's servers.
   - You can close the tab and turn off your PC! It gives you up to 12 hours of uninterrupted GPU time.
   - When it finishes, go to the notebook's **Output tab** to download your trained `model/brahmi_trocr` folder!

### 🔄 How to Resume Training from a Previous Model:
1. **Upload your Weights:** Click **Add Data** → **Upload Data**.
   - Upload your zipped `brahmi_trocr` folder.
   - The notebook will now automatically find it and use it if you set `RESUME_TRAINING = True` below.


## Step 0 — Verify GPU is available

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Detected {torch.cuda.device_count()} GPUs!")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("❌ No GPU detected! Go to Notebook Options → Accelerator → GPU T4 x2")


## Step 1 — Clone Project & Set Working Directory

Kaggle's persistent workspace is `/kaggle/working/`.


In [ ]:
import os
import shutil

PROJECT_DIR = '/kaggle/working/brahmi_ocr_project'

# Clone fresh from GitHub
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone https://github.com/tejaskarade100/brahmi_ocr_project.git {PROJECT_DIR}

%cd {PROJECT_DIR}
!ls


## Step 2 — Install dependencies


In [ ]:
!pip install -q -r requirements.txt

## Step 3a — Link the Kaggle Input Dataset

Kaggle automatically unzips uploaded datasets into the `/kaggle/input/` folder (read-only).
We will locate your dataset and link it directly into our working directory.


In [ ]:
import os
import shutil
import glob

PROJECT_DIR = '/kaggle/working/brahmi_ocr_project'
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')

# Kaggle unzips uploads here. Let's find your 'map.json' to be sure we have the right folder.
input_paths = glob.glob('/kaggle/input/**/map.json', recursive=True)

if not input_paths:
    raise FileNotFoundError("❌ Could not find map.json in /kaggle/input/. Did you upload the 'curated_dataset.zip' using the 'Add Data' button?")

# Use the directory containing the map.json as our true dataset source
KAGGLE_UNZIPPED_DIR = os.path.dirname(input_paths[0])
print(f'📦 Found dataset at: {KAGGLE_UNZIPPED_DIR}')

# Wipe old local dataset dir if it exists to prevent corruption
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

# In Kaggle, we can just natively symlink the /input data directly into our project 
# without wasting time or disk space copying 28,000 images!
os.symlink(KAGGLE_UNZIPPED_DIR, DATASET_DIR)

print('✅ Dataset linked successfully from Kaggle Input!')


In [ ]:
# Validate dataset and generate JSON report
!python dataset/validate_dataset.py --data_dir dataset --json_out dataset_validation_report.json

import json
with open('dataset_validation_report.json', 'r', encoding='utf-8') as f:
    report = json.load(f)

print('Total samples            :', report['total_image_samples'])
print('Mapped classes           :', report['mapped_class_count'])
print('Mapped classes w/ images :', report['mapped_classes_with_images'])
print('Warnings                 :', len(report.get('warnings', [])))
print('Errors                   :', len(report.get('errors', [])))


## Step 3b — Quick class coverage preview


In [ ]:
import json
with open('dataset_validation_report.json', 'r', encoding='utf-8') as f:
    report = json.load(f)

print('Top classes by image count:')
for folder, count in report.get('top_classes_by_count', [])[:10]:
    print(f'  {count:6d}  {folder}')

print('\nBottom classes by image count:')
for folder, count in report.get('bottom_classes_by_count', [])[:10]:
    print(f'  {count:6d}  {folder}')


## Step 3c — Optional strict validation gate


In [ ]:
STRICT_VALIDATE = False  # set True to fail fast on warnings/ratio mismatch

if STRICT_VALIDATE:
    !python dataset/validate_dataset.py --data_dir dataset --strict
else:
    print('Skipping strict gate (STRICT_VALIDATE=False)')


## Step 3d — (Optional) Resume from an Uploaded Model

If you want to continue training your existing model from Google Drive/Kaggle, set `RESUME_FROM_UPLOAD = True`.


In [ ]:
import glob
import os

RESUME_FROM_UPLOAD = False # Set this to True to resume!
MODEL_SEARCH_NAME = 'config.json'
RESUME_MODEL_PATH = 'microsoft/trocr-small-printed' # Default

if RESUME_FROM_UPLOAD:
    # Find any folder in /kaggle/input that contains a 'config.json'
    model_candidates = glob.glob(f'/kaggle/input/**/{MODEL_SEARCH_NAME}', recursive=True)
    
    if model_candidates:
        # Pick the first one we find
        RESUME_MODEL_PATH = os.path.dirname(model_candidates[0])
        print(f'🔄 Resuming from uploaded model found at: {RESUME_MODEL_PATH}')
        print('   (Files found:', os.listdir(RESUME_MODEL_PATH), ')')
    else:
        print('⚠️ No uploaded model found in /kaggle/input/. Falling back to default model.')
else:
    print('🌐 Training from scratch using default model: microsoft/trocr-small-printed')


## Step 4 — Train the TrOCR model 🚀

| Setting | Value |
|---------|-------|
| Base model | `microsoft/trocr-small-printed` |
| Epochs | 15 |
| Batch size | 8 (with Accumulation=4 to equal global batch of 32) |
| Auto-Save | Outputs natively to `/kaggle/working/` |


In [ ]:
!python training/train.py \
    --model_name {RESUME_MODEL_PATH} \
    --epochs 15 \
    --batch_size 8 \
    --gradient_accumulation_steps 4 \
    --balanced_sampling \
    --lr 3e-5 \
    --warmup_ratio 0.1 \
    --patience 4 \
    --data_dir dataset \
    --output_dir /kaggle/working/model/brahmi_trocr \
    --train_ratio 0.8 --val_ratio 0.1 --test_ratio 0.1 \
    --image_size 384


## Step 5 — Test inference on sample images


In [ ]:
# Run inference on first test sample from map-driven dataset
import sys
import os
sys.path.insert(0, '/kaggle/working/brahmi_ocr_project')

MODEL_DIR = '/kaggle/working/model/brahmi_trocr'

if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print(f"❌ Model not found at {MODEL_DIR}. Skipping inference test.")
else:
    from transformers import TrOCRProcessor
    from training.dataset_loader import BrahmiDataset

    processor_for_ds = TrOCRProcessor.from_pretrained(MODEL_DIR, use_fast=False)
    test_ds = BrahmiDataset('dataset', split='test', processor=processor_for_ds, seed=42)

    if len(test_ds.samples) == 0:
        print('No test samples available.')
    else:
        sample = test_ds.samples[0]
        print(f'Test image   : {sample.image_path}')
        print(f'Ground truth : {sample.label_text}')
        !python inference/predict.py --image {sample.image_path} --model_dir {MODEL_DIR} --preprocess --threshold_method auto --debug


In [ ]:
# Batch test: compare predictions vs ground truth
from inference.predict import load_trained_model, predict
from transformers import TrOCRProcessor
from training.dataset_loader import BrahmiDataset
import os

MODEL_DIR = '/kaggle/working/model/brahmi_trocr'

if os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    ds_processor = TrOCRProcessor.from_pretrained(MODEL_DIR, use_fast=False)
    test_ds = BrahmiDataset('dataset', split='test', processor=ds_processor, seed=42)

    processor, model, device = load_trained_model(MODEL_DIR)
    print(f'Model loaded on: {device}\n')

    correct = 0
    total = min(10, len(test_ds.samples))
    for sample in test_ds.samples[:total]:
        result = predict(sample.image_path, processor, model, device, preprocess=False, debug=False)
        pred = result['predicted_text']
        gt = sample.label_text
        match = '✅' if pred.strip() == gt.strip() else '❌'
        if pred.strip() == gt.strip():
            correct += 1
        print(f'{match} {os.path.basename(sample.image_path)}  |  GT: {gt}  |  Pred: {pred}')

    if total > 0:
        print(f'\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)')
else:
    print(f"❌ Model not found at {MODEL_DIR}. Skipping batch test.")


## Done! 🎉

### Your trained model is safely packaged in the Kaggle Output!

If you ran this using **Save & Run All (Commit)**:
1. Go back to the notebook viewer page.
2. Click on the **Output** tab.
3. Download the `model/brahmi_trocr` folder directly to your local PC.

### To resume training later:
1. You can upload the `brahmi_trocr` folder back to Kaggle as a new Input dataset.
2. Change the Base model in `--model_name` to `/kaggle/input/your-model-folder/brahmi_trocr` to resume!

### To run inference locally (PC):
1. Place the `brahmi_trocr` folder into your local `model/` directory.
2. Spin up your backend: `python -m uvicorn main:app`
3. Upload an image in the React UI!
